# Feature Engineering — Predicción Churn Automoción

Pipeline de preparación de datos. Genera `X_model_ready.csv` e `y_model_ready.csv`.

In [1]:
import pandas as pd
import numpy as np
import os

os.chdir(r"C:\Users\alvar\OneDrive\Escritorio\Ingeniería Matemática\3º de ing Matemática\Inteligencia Artificial\Churn\RESOLUCIÓN")

url = "https://storage.googleapis.com/jvelare-public/bq-results-20260211-093843-1770799837095"
df = pd.read_csv(url)
df.columns = df.columns.str.strip()

print("Shape original:", df.shape)
print("Columnas:", df.columns.tolist())


Shape original: (58049, 40)
Columnas: ['CODE', 'Sales_Date', 'Id_Producto', 'Customer_ID', 'PVP', 'MOTIVO_VENTA', 'FORMA_PAGO', 'EXTENSION_GARANTIA', 'SEGURO_BATERIA_LARGO_PLAZO', 'MANTENIMIENTO_GRATUITO', 'FIN_GARANTIA', 'BASE_DATE', 'EN_GARANTIA', 'COSTE_VENTA_NO_IMPUESTOS', 'GENERO', 'CODIGO_POSTAL', 'Edad', 'RENTA_MEDIA_ESTIMADA', 'ENCUESTA_CLIENTE_ZONA_TALLER', 'STATUS_SOCIAL', 'Modelo', 'TIPO_CARROCERIA', 'Fuel', 'TRANSMISION_ID', 'Equipamiento', 'Kw', 'Revisiones', 'Km_medio_por_revision', 'km_ultima_revision', 'QUEJA', 'Lead_compra', 'Fue_Lead', 'TIENDA_DESC', 'PROV_DESC', 'ZONA', 'Origen', 'Margen_eur_bruto', 'Margen_eur', 'DAYS_LAST_SERVICE', 'Churn_400']


In [2]:
# ── 2. Variable objetivo ─────────────────────────────────────────────────────
TARGET = "Churn_400"

df[TARGET] = (
    df[TARGET].astype("string").str.strip().str.upper()
    .map({"Y": 1, "N": 0})
)

y = df[TARGET].copy()
print("Distribución target:")
print(y.value_counts())
print(f"Churn rate: {y.mean()*100:.2f}%")


Distribución target:
Churn_400
0    52956
1     5093
Name: count, dtype: int64
Churn rate: 8.77%


In [3]:
# ── 3. Eliminar variables ─────────────────────────────────────────────────────
# Eliminamos:
# - Identificadores (no predicen nada)
# - Leakage real: DAYS_LAST_SERVICE (define el churn)
# - Financieras post-venta: Margen_eur_bruto, Margen_eur
# - Variables de taller: no disponibles en clientes nuevos
# - days_since_sale: clientes nuevos tienen distinta antigüedad
# - EN_GARANTIA: redundante con days_to_warranty_end

cols_drop = [
    TARGET,
    "Customer_ID", "CODE", "Id_Producto",
    "DAYS_LAST_SERVICE",
    "Margen_eur_bruto", "Margen_eur",
    "EN_GARANTIA",
    "Revisiones", "Km_medio_por_revision", "km_ultima_revision",
    "days_since_sale",
]
cols_drop = [c for c in cols_drop if c in df.columns]
X = df.drop(columns=cols_drop)

print("Eliminadas:", cols_drop)
print("Shape:", X.shape)


Eliminadas: ['Churn_400', 'Customer_ID', 'CODE', 'Id_Producto', 'DAYS_LAST_SERVICE', 'Margen_eur_bruto', 'Margen_eur', 'EN_GARANTIA', 'Revisiones', 'Km_medio_por_revision', 'km_ultima_revision']
Shape: (58049, 29)


In [4]:
# ── 4. Fechas → días ──────────────────────────────────────────────────────────
for col in ["Sales_Date", "BASE_DATE", "FIN_GARANTIA"]:
    if col in X.columns:
        X[col] = pd.to_datetime(X[col], errors="coerce", dayfirst=True)

if "FIN_GARANTIA" in X.columns and "BASE_DATE" in X.columns:
    X["days_to_warranty_end"] = (X["FIN_GARANTIA"] - X["BASE_DATE"]).dt.days
    print("✅ days_to_warranty_end creada")

if "days_to_warranty_end" in X.columns:
    X["warranty_expired"] = (X["days_to_warranty_end"] < 0).astype(int)
    print("✅ warranty_expired creada")

X = X.drop(columns=[c for c in ["Sales_Date","BASE_DATE","FIN_GARANTIA"] if c in X.columns])
print("Shape:", X.shape)


✅ days_to_warranty_end creada
✅ warranty_expired creada
Shape: (58049, 28)


In [5]:
# ── 5. Nuevas variables de perfil y compromiso ───────────────────────────────

# Score de compromiso: cuántos productos de fidelización contrató (0-3)
ext_garantia = (df["EXTENSION_GARANTIA"].str.upper().str.startswith("SI")).astype(int)
seguro_bat   = (df["SEGURO_BATERIA_LARGO_PLAZO"].str.upper() == "SI").astype(int)
mant_gratis  = (df["MANTENIMIENTO_GRATUITO"] > 0).astype(int)
X["compromiso_score"] = ext_garantia + seguro_bat + mant_gratis
print("✅ compromiso_score (0-3)")

# Esfuerzo financiero: PVP / Renta → sensibilidad al precio del taller
renta_safe = df["RENTA_MEDIA_ESTIMADA"].replace(0, df["RENTA_MEDIA_ESTIMADA"].median())
X["esfuerzo_financiero"] = df["PVP"] / renta_safe
print("✅ esfuerzo_financiero (PVP/Renta)")

# Garantía expira pronto: menos de 6 meses → momento crítico de churn
X["garantia_expira_pronto"] = (
    (X["days_to_warranty_end"] >= 0) & (X["days_to_warranty_end"] < 180)
).astype(int)
print("✅ garantia_expira_pronto (<180 días)")

# Financiado por la marca → mayor vínculo con el concesionario
X["es_financiado_marca"] = (
    df["FORMA_PAGO"].str.upper().str.contains("FINANCIERA MARCA", na=False)
).astype(int)
print("✅ es_financiado_marca")

# Segmento premium
X["is_luxury_segment"] = (df["PVP"] > 35000).astype(int)
print("✅ is_luxury_segment")

print("\nShape:", X.shape)


✅ compromiso_score (0-3)
✅ esfuerzo_financiero (PVP/Renta)
✅ garantia_expira_pronto (<180 días)
✅ es_financiado_marca
✅ is_luxury_segment

Shape: (58049, 33)


In [6]:
# ── 6. Eliminar alta granularidad ─────────────────────────────────────────────
cols_hg = ["CODIGO_POSTAL", "TIENDA_DESC"]
cols_hg = [c for c in cols_hg if c in X.columns]
X = X.drop(columns=cols_hg)
print("Eliminadas por alta granularidad:", cols_hg)
print("Shape:", X.shape)


Eliminadas por alta granularidad: ['CODIGO_POSTAL', 'TIENDA_DESC']
Shape: (58049, 31)


In [7]:
# ── 7. Imputar nulos ──────────────────────────────────────────────────────────
print("Nulos antes:")
nulos = X.isnull().sum()
print(nulos[nulos > 0].sort_values(ascending=False))

for col in X.select_dtypes(include=["int64","float64"]).columns:
    X[col] = X[col].fillna(X[col].median())
for col in X.select_dtypes(include="object").columns:
    X[col] = X[col].fillna("Unknown")

print(f"\nNulos después: {X.isnull().sum().sum()}")


Nulos antes:
QUEJA            33323
STATUS_SOCIAL    12816
GENERO             849
dtype: int64

Nulos después: 0


In [8]:
# ── 8. One-Hot Encoding ───────────────────────────────────────────────────────
cat_cols = X.select_dtypes(include="object").columns.tolist()
print("Variables a codificar:", cat_cols)

X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
print("\nShape final de X:", X.shape)


Variables a codificar: ['MOTIVO_VENTA', 'FORMA_PAGO', 'EXTENSION_GARANTIA', 'SEGURO_BATERIA_LARGO_PLAZO', 'GENERO', 'STATUS_SOCIAL', 'Modelo', 'TIPO_CARROCERIA', 'Fuel', 'TRANSMISION_ID', 'Equipamiento', 'QUEJA', 'PROV_DESC', 'ZONA', 'Origen']

Shape final de X: (58049, 72)


In [9]:
# ── 9. Resumen final ──────────────────────────────────────────────────────────
print(f"Shape final → {X.shape[0]} filas x {X.shape[1]} variables\n")
for idx, col in enumerate(X.columns, 1):
    print(f"  {idx:>3}. {col}")

print("\n✅ Nuevas variables:")
for v in ["compromiso_score","esfuerzo_financiero","garantia_expira_pronto","es_financiado_marca"]:
    print(f"  {'✅' if v in X.columns else '❌'} {v}")


Shape final → 58049 filas x 72 variables

    1. PVP
    2. MANTENIMIENTO_GRATUITO
    3. COSTE_VENTA_NO_IMPUESTOS
    4. Edad
    5. RENTA_MEDIA_ESTIMADA
    6. ENCUESTA_CLIENTE_ZONA_TALLER
    7. Kw
    8. Lead_compra
    9. Fue_Lead
   10. days_to_warranty_end
   11. warranty_expired
   12. compromiso_score
   13. esfuerzo_financiero
   14. garantia_expira_pronto
   15. es_financiado_marca
   16. is_luxury_segment
   17. MOTIVO_VENTA_Particular
   18. FORMA_PAGO_Financiera Marca
   19. FORMA_PAGO_Otros
   20. FORMA_PAGO_Prestamo Bancario
   21. EXTENSION_GARANTIA_SI
   22. EXTENSION_GARANTIA_SI, Campa a Regalo
   23. EXTENSION_GARANTIA_SI, Financiera
   24. SEGURO_BATERIA_LARGO_PLAZO_SI
   25. GENERO_M
   26. GENERO_Unknown
   27. STATUS_SOCIAL_B
   28. STATUS_SOCIAL_C
   29. STATUS_SOCIAL_D
   30. STATUS_SOCIAL_E
   31. STATUS_SOCIAL_F
   32. STATUS_SOCIAL_G
   33. STATUS_SOCIAL_H
   34. STATUS_SOCIAL_I
   35. STATUS_SOCIAL_J
   36. STATUS_SOCIAL_K
   37. STATUS_SOCIAL_U
   38. STA

In [10]:
# ── 10. Exportar ──────────────────────────────────────────────────────────────
X.to_csv("X_model_ready.csv", index=False)
y.to_csv("y_model_ready.csv", index=False)

print(f"✅ X_model_ready.csv → {X.shape}")
print(f"✅ y_model_ready.csv → {y.shape}")
print(f"📁 Carpeta: {os.getcwd()}")


✅ X_model_ready.csv → (58049, 72)
✅ y_model_ready.csv → (58049,)
📁 Carpeta: C:\Users\alvar\OneDrive\Escritorio\Ingeniería Matemática\3º de ing Matemática\Inteligencia Artificial\Churn\RESOLUCIÓN
